# Phase 5 — Model 4 — KD — matching your uploaded notebook
Based on notebook93c54d7f80.ipynb. Preserves its full train/dev data, 3 x 192-token views, CHARS_PER_VIEW=2400, three epochs, batch 2 / accumulation 4, simple mean/last-token head, positive-weight cap 8, and development threshold grid. No deadline subsets or v5 head changes. Development scores are threshold-tuned and checkpoint-selected, not independent test scores. Runtime and F1 are unverified for these variants.
M4 KD requires best_model.pt produced by that exact uploaded M4 LoRMA notebook, not the later balanced/v5 notebooks. Set TEACHER_CKPT_PATH. Student has 8 layers; teacher stays frozen.


### 1. Configuration and imports


In [11]:
import os, sys, json, re, math, time, random, hashlib, zipfile, gc, warnings
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict
from contextlib import nullcontext
from functools import lru_cache
import numpy as np
import torch
import torch.utils.checkpoint
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

MODEL_ID = 4
METHOD = 'kd'
REPO_ID = 'Exploration-Lab/CS-781-Capstone'
# Only YOUR original pretrained Phase-1 weights and BPE are valid here.
BASE_CKPT_PATH = ''     # optionally /kaggle/input/.../pytorch_model.bin
BASE_TOKENIZER_PATH = '' # optionally /kaggle/input/.../custom_bpe_scratch.json
TEACHER_CKPT_PATH = '/kaggle/input/datasets/kalyankakarapathi/phase5m4lormamodel/best_model.pt'  # KD only: Phase-5 Model-N LORMA best_model.pt
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = OUTPUT_ROOT / f'phase5_lsi_model{MODEL_ID}_{METHOD}_reference'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 32000
N_EMBD = 512 if MODEL_ID == 2 else 560
N_LAYER = 15
N_HEAD = 8
N_KV_HEAD = 4
MAX_SEQ_LEN = 2048
MAX_CHUNKS = 3
TOKENS_PER_CHUNK = 192
CHARS_PER_VIEW = 2400
BATCH_SIZE = 4
GRAD_ACCUM = 2
EPOCHS = 3
PATIENCE = 2
LR_BACKBONE = 1.5e-5
LR_HEAD = 2e-4
LR_ADAPTER = 1.5e-4
WEIGHT_DECAY = 0.01
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16
DROPOUT = 0.10
STUDENT_LAYERS = 8
KD_TEMPERATURE = 2.0
KD_SOFT_WEIGHT = 0.5
GRAD_CLIP = 1.0
WARMUP_FRACTION = 0.06
NUM_WORKERS = 0
# The maximum positive weight is capped to avoid destabilization from very rare statute IDs.
MAX_POS_WEIGHT = 8.0
MIN_POS_WEIGHT = 1.0
MIN_DEV_POSITIVES_FOR_PER_LABEL_TUNING = 3
THRESHOLD_GRID = np.arange(0.10, 0.91, 0.05)
CHECKPOINT_GRADIENTS = True
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
print('Running', MODEL_ID, METHOD, '| device', DEVICE, '| output', OUTPUT_DIR)


Running 4 kd | device cuda | output /kaggle/working/phase5_lsi_model4_kd_reference


### 2. Download official Phase-5 data securely


In [12]:
# Hugging Face token is read from Kaggle Secrets. Never print or hard-code it.
def get_hf_token():
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF')
    except Exception as exc:
        raise RuntimeError('Missing Hugging Face token: enable Kaggle Secret HF or HF_TOKEN.') from exc

HF_TOKEN = get_hf_token()

def download_json(path):
    try:
        local = hf_hub_download(repo_id=REPO_ID, repo_type='dataset', filename=path, token=HF_TOKEN)
    except Exception as exc:
        raise RuntimeError(f'Cannot download {path} from {REPO_ID}: verify dataset permission and Internet.') from exc
    with open(local, encoding='utf-8') as handle:
        return json.load(handle)

raw_train = download_json('Phase_5_LSI/Train/train.json')
raw_dev = download_json('Phase_5_LSI/Train/dev.json')
raw_test = download_json('Phase_5_LSI/Test/test.json')
print('Downloaded the three official Phase-5 JSON files.')


Downloaded the three official Phase-5 JSON files.


### 3. Validate IDs, aggregate multi-label duplicates, preserve official test order


In [13]:
# Preserve original case IDs, original statutes, and multiple labels per case.
# No outside legal data, no gold test labels, and no model-generated substitute labels.

def to_records(obj, split):
    if isinstance(obj, list): return obj
    if isinstance(obj, dict):
        if isinstance(obj.get('data'), list): return obj['data']
        # The document shows a list of objects; ID-keyed dictionaries are supported conservatively.
        if all(isinstance(v, dict) for v in obj.values()):
            return [dict(v, id=v.get('id',k)) for k,v in obj.items()]
    raise ValueError(f'{split}: expected list of records or ID-to-record dictionary; got {type(obj)}')

def canonical_labels(value, split, ident):
    if not isinstance(value, list):
        raise ValueError(f'{split} {ident}: labels must be a list of integer statute IDs')
    found=[]
    for v in value:
        if isinstance(v, bool) or not isinstance(v, (int, np.integer)):
            raise ValueError(f'{split} {ident}: invalid statute ID {v!r}; integers only')
        if int(v) < 0: raise ValueError(f'{split} {ident}: negative statute ID {v}')
        found.append(int(v))
    return sorted(set(found))

def clean_text(value):
    if not isinstance(value, str): return ''
    value = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

def prepare(obj, split, labeled):
    groups=OrderedDict()
    errors=[]
    for idx, row in enumerate(to_records(obj, split)):
        if not isinstance(row, dict):
            errors.append((idx,'not an object')); continue
        rid=row.get('id')
        if isinstance(rid,bool) or not isinstance(rid,(str,int)) or not str(rid).strip():
            raise ValueError(f'{split}[{idx}]: invalid case id: {rid!r}')
        ident=str(rid)
        t=clean_text(row.get('text',''))
        if not t and split != 'test':
            errors.append((idx,'empty text')); continue
        if not t: t=' ' # do not omit official test ID
        labels=canonical_labels(row.get('labels'),split,ident) if labeled else []
        if ident not in groups:
            groups[ident]={'id':rid,'text':t,'segments':[],'labels':set()}
        g=groups[ident]
        if t not in g['segments']: g['segments'].append(t)
        # Duplicate case records can enumerate different applicable statute IDs; union them.
        g['labels'].update(labels)
    records=[]
    for g in groups.values():
        g['segments']=sorted(g['segments'],key=len,reverse=True)
        g['text']=g['segments'][0]
        g['labels']=sorted(g['labels'])
        records.append(g)
    print(f'{split}: {len(records):,} distinct IDs; raw rows {len(to_records(obj,split)):,}; rejected {len(errors)}')
    if not records: raise ValueError(f'{split}: empty after cleaning')
    return records

train_records=prepare(raw_train,'train',True)
dev_records=prepare(raw_dev,'dev',True)
test_records=prepare(raw_test,'test',False)

# Cross-split ID leakage control (not new labeled-data augmentation).
test_ids={str(r['id']) for r in test_records}
dev_records=[r for r in dev_records if str(r['id']) not in test_ids]
dev_ids={str(r['id']) for r in dev_records}
train_records=[r for r in train_records if str(r['id']) not in test_ids and str(r['id']) not in dev_ids]
assert train_records and dev_records, 'Train/dev empty after ID leakage audit'

train_seen={s for r in train_records for s in r['labels']}
dev_seen={s for r in dev_records for s in r['labels']}
label_ids=sorted(train_seen|dev_seen)
# The document gives no separate statutory-label vocabulary; these IDs are observed on official public data.
assert label_ids and all(type(i)==int for i in label_ids)
label_to_col={v:i for i,v in enumerate(label_ids)}
NUM_LABELS=len(label_ids)
print('Statute IDs',NUM_LABELS,'| seen in train',len(train_seen),
      '| development-only',len(dev_seen-train_seen))
print('Cases with multiple labels:',sum(len(r['labels'])>1 for r in train_records))
print('Top train labels:',Counter(s for r in train_records for s in r['labels']).most_common(12))
print('CAUTION: zero training positives for development-only IDs:',sorted(dev_seen-train_seen)[:25])
(OUTPUT_DIR/'label_ids.json').write_text(json.dumps(label_ids),encoding='utf-8')


train: 42,750 distinct IDs; raw rows 42,750; rejected 0
dev: 10,181 distinct IDs; raw rows 10,181; rejected 0
test: 13,019 distinct IDs; raw rows 13,019; rejected 0
Statute IDs 100 | seen in train 100 | development-only 0
Cases with multiple labels: 33604
Top train labels: [(5, 11730), (34, 10380), (97, 7880), (43, 7728), (39, 7120), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835), (14, 3818), (61, 3723)]
CAUTION: zero training positives for development-only IDs: []


390

### 4. Original Phase-1 model architecture (same pretrained parameter keys)


In [14]:
# Exact Phase-1 Model 4 module/key layout reused from the user's earlier experiment.
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__(); self.eps=eps; self.weight=nn.Parameter(torch.ones(dim))
    def forward(self,x):
        return x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+self.eps)*self.weight

def precompute_rope(dim,max_len,theta=10000.0):
    f=1.0/(theta**(torch.arange(0,dim,2).float()/dim))
    xy=torch.outer(torch.arange(max_len),f)
    return torch.view_as_real(torch.polar(torch.ones_like(xy),xy))

def apply_rope(q,k,f):
    qc=torch.view_as_complex(q.float().reshape(*q.shape[:-1],-1,2))
    kc=torch.view_as_complex(k.float().reshape(*k.shape[:-1],-1,2))
    z=f.view(1,qc.shape[1],1,qc.shape[-1])
    return torch.view_as_real(qc*z).flatten(3).type_as(q),torch.view_as_real(kc*z).flatten(3).type_as(k)

class SinkWindowAttention(nn.Module):
    def __init__(self,head_dim,window_size=256,sink_size=4):
        super().__init__();self.scale=head_dim**(-0.5)
        self.window_size=window_size;self.sink_size=sink_size
    def create_sink_window_mask(self,Lq,Lk,device):
        row=torch.arange(Lq,device=device).unsqueeze(1)
        col=torch.arange(Lk,device=device).unsqueeze(0)
        m=(col<=row)&((row-col)<self.window_size)
        if self.sink_size:m=m|((col<self.sink_size)&(col<=row))
        return m[None,None,:,:]
    def forward(self,q,k,v,is_causal=True,attention_mask=None):
        Lq,Lk=q.shape[2],k.shape[2]
        scores=(q@k.transpose(-2,-1))*self.scale
        if is_causal and Lq>1:
            scores=scores.masked_fill(~self.create_sink_window_mask(Lq,Lk,q.device),
                                       torch.finfo(scores.dtype).min)
        if attention_mask is not None:
            scores=scores.masked_fill(~attention_mask[:,None,None,:].bool(),torch.finfo(scores.dtype).min)
        return torch.softmax(scores,dim=-1)@v

class EfficientGQA(nn.Module):
    def __init__(self,dim,n_heads,n_kv_heads,window_size=256,sink_size=4):
        super().__init__()
        self.n_heads=n_heads;self.n_kv_heads=n_kv_heads;self.head_dim=dim//n_heads
        self.wq=nn.Linear(dim,n_heads*self.head_dim,bias=False)
        self.wk=nn.Linear(dim,n_kv_heads*self.head_dim,bias=False)
        self.wv=nn.Linear(dim,n_kv_heads*self.head_dim,bias=False)
        self.wo=nn.Linear(dim,dim,bias=False)
        self.attention=SinkWindowAttention(self.head_dim,window_size,sink_size)
    def forward(self,x,freqs_cis,kv_cache=None,attention_mask=None):
        B,L,_=x.shape
        q=self.wq(x).view(B,L,self.n_heads,self.head_dim)
        k=self.wk(x).view(B,L,self.n_kv_heads,self.head_dim)
        v=self.wv(x).view(B,L,self.n_kv_heads,self.head_dim)
        q,k=apply_rope(q,k,freqs_cis)
        if kv_cache is not None:
            kc,vc=kv_cache;k=torch.cat((kc,k),dim=1);v=torch.cat((vc,v),dim=1)
        new_cache=(k,v)
        k=k.repeat_interleave(self.n_heads//self.n_kv_heads,dim=2)
        v=v.repeat_interleave(self.n_heads//self.n_kv_heads,dim=2)
        out=self.attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                           is_causal=(L>1),attention_mask=attention_mask)
        return self.wo(out.transpose(1,2).contiguous().view(B,L,-1)),new_cache

class StandardFFN(nn.Module):
    def __init__(self,dim):
        super().__init__();self.net=nn.Sequential(nn.Linear(dim,dim*4,bias=False),nn.GELU(),
                                                  nn.Linear(dim*4,dim,bias=False))
    def forward(self,x):return self.net(x)
class BottleneckFFN(nn.Module):
    def __init__(self,dim):
        super().__init__();self.net=nn.Sequential(nn.Linear(dim,dim//4,bias=False),nn.GELU(),
                                                  nn.Linear(dim//4,dim,bias=False))
    def forward(self,x):return self.net(x)
class EfficientTransformerBlock(nn.Module):
    def __init__(self,dim,n_heads,n_kv_heads,window_size=256,sink_size=4,is_bottleneck=False):
        super().__init__()
        self.norm1=RMSNorm(dim);self.norm2=RMSNorm(dim)
        self.attn=EfficientGQA(dim,n_heads,n_kv_heads,window_size,sink_size)
        self.ffn=BottleneckFFN(dim) if is_bottleneck else StandardFFN(dim)
    def forward(self,x,freqs_cis,kv_cache=None,attention_mask=None):
        out,c=self.attn(self.norm1(x),freqs_cis,kv_cache,attention_mask)
        h=x+out
        return h+self.ffn(self.norm2(h)),c
class CustomLegalModel(nn.Module):
    def __init__(self,vocab_size,dim,n_layers,n_heads,n_kv_heads,max_len,window_size=256,sink_size=4):
        super().__init__()
        self.embeddings=nn.Embedding(vocab_size,dim)
        self.layers=nn.ModuleList([
            EfficientTransformerBlock(dim,n_heads,n_kv_heads,window_size,sink_size,i%2!=0)
            for i in range(n_layers)])
        self.norm=RMSNorm(dim)
        self.output=nn.Linear(dim,vocab_size,bias=False)
        self.output.weight=self.embeddings.weight
        self.register_buffer('freqs_cis',precompute_rope(dim//n_heads,max_len))
    def forward(self,tokens,start_pos=0,kv_caches=None,attention_mask=None):
        _,L=tokens.shape
        freqs=torch.view_as_complex(self.freqs_cis[start_pos:start_pos+L].to(tokens.device))
        x=self.embeddings(tokens);caches=[]
        for i,layer in enumerate(self.layers):
            x,c=layer(x,freqs,kv_caches[i] if kv_caches is not None else None,attention_mask)
            caches.append(c)
        return self.output(self.norm(x)),caches


### 5. Load original byte-level BPE and verify pretrained weights strictly


In [15]:
# Original byte-level BPE's serialized merges must be used; never fit a new BPE on Phase 5.
class CustomByteLevelBPE:
    def __init__(self,vocab_size=32000):
        self.vocab_size=vocab_size;self.merges={};self.vocab={i:bytes([i]) for i in range(256)}
        self.pad_id=0
    def load(self,path):
        with open(path,encoding='utf-8') as f:d=json.load(f)
        self.vocab_size=d['vocab_size'];self.next_id=d['next_id']
        self.merges={tuple(map(int,k.split(','))):int(v) for k,v in d['merges'].items()}
        self.vocab={int(k):bytes(v) for k,v in d['vocab'].items()}
        return self
    def encode(self,text):
        output=[]
        for chunk in re.findall(r'\S+|\s+',text):
            ids=list(chunk.encode('utf-8'))
            while len(ids)>1:
                pairs={(ids[i],ids[i+1]) for i in range(len(ids)-1)}
                pair=min(pairs,key=lambda p:self.merges.get(p,float('inf')))
                if pair not in self.merges:break
                out=[];j=0
                while j<len(ids):
                    if j<len(ids)-1 and (ids[j],ids[j+1])==pair:
                        out.append(self.merges[pair]);j+=2
                    else:out.append(ids[j]);j+=1
                ids=out
            output.extend(ids)
        return output
    def decode(self,ids):
        return b''.join(self.vocab[i] for i in ids if i!=self.pad_id).decode('utf-8','replace')

def find_asset(explicit,filename,model_id):
    if explicit:
        p=Path(explicit)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    paths=[]
    for root in (Path('/kaggle/input'),Path('/kaggle/working')):
        if root.exists():paths+=list(root.rglob(filename))
    paths=sorted(set(p for p in paths if p.is_file()))
    if len(paths)==1:return paths[0]
    hints=[f'model{model_id}',f'model_{model_id}',f'm{model_id}_']
    good=[p for p in paths if any(h in str(p).lower() for h in hints)]
    if len(good)==1:return good[0]
    raise FileNotFoundError(f'Cannot uniquely choose {filename} for Model {model_id}. '
                            f'Set explicit config path. Matches: {[str(p) for p in paths]}')

base_path=find_asset(BASE_CKPT_PATH,'pytorch_model.bin',MODEL_ID)
tokenizer_path=find_asset(BASE_TOKENIZER_PATH,'custom_bpe_scratch.json',MODEL_ID)
tokenizer=CustomByteLevelBPE().load(tokenizer_path)
assert len(tokenizer.merges)>0 and max(tokenizer.vocab)<VOCAB_SIZE

def load_p1_state(path):
    saved=torch.load(path,map_location='cpu',weights_only=True)
    for key in ('model_state_dict','state_dict'):
        if key in saved and isinstance(saved[key],dict):saved=saved[key]
    assert isinstance(saved,dict)
    return {k.removeprefix('module.'):v for k,v in saved.items()}
BASE_STATE=load_p1_state(base_path)
emb_name='tok_embeddings.weight' if MODEL_ID==2 else 'embeddings.weight'
assert emb_name in BASE_STATE and tuple(BASE_STATE[emb_name].shape)==(VOCAB_SIZE,N_EMBD), (
    f'Wrong Model-{MODEL_ID} checkpoint: {emb_name} mismatch')

def new_base(num_layers=N_LAYER):
    model=CustomLegalModel(VOCAB_SIZE,N_EMBD,num_layers,N_HEAD,N_KV_HEAD,MAX_SEQ_LEN)
    # When creating a compressed student, initialize every matching layer and shared embedding.
    desired=model.state_dict()
    bad=[k for k,v in desired.items() if k not in BASE_STATE or BASE_STATE[k].shape!=v.shape]
    if bad:raise RuntimeError(f'Phase-1 state mismatch {bad[:12]}')
    model.load_state_dict({k:BASE_STATE[k] for k in desired},strict=True)
    return model

print('Verified exact Model',MODEL_ID,'Phase-1 checkpoint:',base_path)
print('Tokenizer merges:',len(tokenizer.merges))


Verified exact Model 4 Phase-1 checkpoint: /kaggle/input/datasets/kalyankakarapathi/phase1m4/pytorch_model.bin
Tokenizer merges: 31744


### 6. Adaptation modules and multi-label classification head


In [16]:
class LowRankAdditive(nn.Module):
    # LoRA: delta W = B @ A, evaluated on the input of frozen W0.
    def __init__(self,base,r=8,alpha=16,dropout=0.05):
        super().__init__();self.base=base;self.base.requires_grad_(False)
        self.A=nn.Linear(base.in_features,r,bias=False)
        self.B=nn.Linear(r,base.out_features,bias=False)
        self.drop=nn.Dropout(dropout);self.scale=alpha/r
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5));nn.init.zeros_(self.B.weight)
    def forward(self,x):return self.base(x)+self.scale*self.B(self.A(self.drop(x)))

class LowRankMultiplicative(nn.Module):
    # LORMA-style post-projection low-rank multiplicative update:
    # (I + alpha/r * BA) W0 x, distinct from additive LoRA on x.
    def __init__(self,base,r=8,alpha=16,dropout=0.05):
        super().__init__();self.base=base;self.base.requires_grad_(False)
        self.A=nn.Linear(base.out_features,r,bias=False)
        self.B=nn.Linear(r,base.out_features,bias=False)
        self.drop=nn.Dropout(dropout);self.scale=alpha/r
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5));nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.base(x);return y+self.scale*self.B(self.A(self.drop(y)))

def adapt_backbone(backbone,method):
    if method in ('full','student_full'):
        backbone.requires_grad_(True)
        return
    if method not in ('lora','lorma'):
        raise ValueError('Expected full, lora, lorma or student_full; got '+method)
    backbone.requires_grad_(False)
    target=LowRankMultiplicative if method=='lorma' else LowRankAdditive
    for block in backbone.layers:
        for key in ('wq','wk','wv','wo'):
            original=getattr(block.attn,key)
            setattr(block.attn,key,target(original,ADAPTER_RANK,ADAPTER_ALPHA,DROPOUT))

class MultiLabelLegalClassifier(nn.Module):
    def __init__(self,backbone,num_labels,method='full'):
        super().__init__()
        self.backbone=backbone
        adapt_backbone(self.backbone,method)
        # Pool masked tokens -> mean of chunk representations -> multi-label logits.
        self.head=nn.Sequential(nn.LayerNorm(N_EMBD*2),nn.Dropout(DROPOUT),
                                nn.Linear(N_EMBD*2,num_labels))
        self.num_labels=num_labels
    def forward(self,input_ids,token_mask,chunk_mask):
        B,K,L=input_ids.shape
        ids=input_ids.view(B*K,L)
        mask=token_mask.view(B*K,L)
        base=self.backbone
        freqs=torch.view_as_complex(base.freqs_cis[:L].to(ids.device))
        x=(base.tok_embeddings(ids) if MODEL_ID==2 else base.embeddings(ids))
        for layer in base.layers:
            if CHECKPOINT_GRADIENTS and self.training and x.requires_grad:
                def call(h,layer=layer):
                    return layer(h,freqs,attention_mask=mask)[0]
                x=torch.utils.checkpoint.checkpoint(call,x,use_reentrant=False)
            else:
                x,_=layer(x,freqs,attention_mask=mask)
        x=base.norm(x)
        active=mask.unsqueeze(-1).to(x.dtype)
        mean=(x*active).sum(1)/active.sum(1).clamp(min=1)
        # For right-padded sequences: final valid token (causally aware document vector).
        final=(mask.long().sum(1)-1).clamp(min=0)
        last=x[torch.arange(x.size(0),device=x.device),final]
        per_chunk=torch.cat((mean,last),dim=-1).view(B,K,-1)
        weights=chunk_mask.to(per_chunk.dtype).unsqueeze(-1)
        doc=(per_chunk*weights).sum(1)/weights.sum(1).clamp(min=1)
        return self.head(doc)


### 7. Statute-aware long-document views, dataset, positive weighting


In [17]:
STATUTE_CUE=re.compile(r'(?i)\b(?:sections?|articles?|acts?|statutes?|constitution|rules?|\d+\s+of\s+\d{4})\b')

def select_views(row):
    segments=row.get('segments') or [row['text']]
    main=segments[0]
    n=CHARS_PER_VIEW
    # Front/back and a citation-heavy window: labels are never consulted for view selection.
    views=[main[:n]]
    if len(main)>n:views.append(main[-n:])
    matches=list(STATUTE_CUE.finditer(main))
    if matches:
        mid=matches[len(matches)//2].start()
        views.append(main[max(0,mid-n//2):max(0,mid-n//2)+n])
    else:
        mid=max(0,(len(main)-n)//2)
        views.append(main[mid:mid+n])
    for part in segments[1:]:views.append(part[:n])
    unique=list(dict.fromkeys(v for v in views if v))
    return unique[:MAX_CHUNKS] or [' ']

@lru_cache(maxsize=25000)
def encode_view(s):
    x=tokenizer.encode(s)[:TOKENS_PER_CHUNK]
    if not x:x=[32] # safe byte-space fallback for empty legal text
    return x

class CaseDataset(Dataset):
    def __init__(self,records):self.records=records
    def __len__(self):return len(self.records)
    def __getitem__(self,i):
        row=self.records[i]
        ids=torch.zeros(MAX_CHUNKS,TOKENS_PER_CHUNK,dtype=torch.long)
        mask=torch.zeros(MAX_CHUNKS,TOKENS_PER_CHUNK,dtype=torch.bool)
        cmask=torch.zeros(MAX_CHUNKS,dtype=torch.bool)
        for j,v in enumerate(select_views(row)):
            seq=encode_view(v)
            if not all(0<=x<VOCAB_SIZE for x in seq):
                raise ValueError(f'Tokenizer generated invalid token ID in case {row["id"]}')
            ids[j,:len(seq)]=torch.tensor(seq)
            mask[j,:len(seq)]=True
            cmask[j]=True
        y=torch.zeros(NUM_LABELS,dtype=torch.float32)
        for label in row.get('labels',[]):y[label_to_col[label]]=1
        return ids,mask,cmask,y

train_dl=DataLoader(CaseDataset(train_records),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
dev_dl=DataLoader(CaseDataset(dev_records),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
test_dl=DataLoader(CaseDataset(test_records),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)

positive_counts=np.zeros(NUM_LABELS,dtype=np.float32)
for r in train_records:
    for s in r['labels']:positive_counts[label_to_col[s]]+=1
# For absent positives, do not make the all-negative class dominate training.
weights=np.clip((len(train_records)-positive_counts)/np.maximum(positive_counts,1),MIN_POS_WEIGHT,MAX_POS_WEIGHT)
weights[positive_counts==0]=1
pos_weight=torch.tensor(weights,device=DEVICE,dtype=torch.float32)

def multilabel_loss(logits,targets):
    return F.binary_cross_entropy_with_logits(logits.float(),targets.float(),pos_weight=pos_weight)

print('Train examples',len(train_records),'dev',len(dev_records),'test',len(test_records))


Train examples 42750 dev 10181 test 13019


### 8. Exact public DEV Macro-F1, Micro-F1, subset accuracy, threshold selection


In [18]:
def metrics_for(y_true,y_prob,thresholds):
    y_hat=(y_prob>=np.asarray(thresholds)[None,:]).astype('int8')
    # Keep all label columns; Macro-F1 includes rare labels with zero support.
    indices=np.arange(NUM_LABELS)
    report={
        'macro_f1':float(f1_score(y_true,y_hat,average='macro',labels=indices,zero_division=0)),
        'micro_f1':float(f1_score(y_true,y_hat,average='micro',labels=indices,zero_division=0)),
        'subset_accuracy':float(accuracy_score(y_true,y_hat)),
        'macro_precision':float(precision_score(y_true,y_hat,average='macro',labels=indices,zero_division=0)),
        'macro_recall':float(recall_score(y_true,y_hat,average='macro',labels=indices,zero_division=0)),
        'predicted_labels_per_case':float(y_hat.sum(1).mean()),
    }
    return report

def tune_thresholds(y_true,y_prob):
    # All choices are made on PUBLIC DEV only, never on the official private test.
    # First choose one global threshold; then allow stable dev-positive labels individual adjustment.
    best_global=0.5;best_score=-1
    for th in THRESHOLD_GRID:
        score=metrics_for(y_true,y_prob,np.full(NUM_LABELS,th))['macro_f1']
        if score>best_score+1e-12:
            best_global=float(th);best_score=score
    ts=np.full(NUM_LABELS,best_global,dtype=np.float64)
    for j in range(NUM_LABELS):
        if y_true[:,j].sum()<MIN_DEV_POSITIVES_FOR_PER_LABEL_TUNING:continue
        best_t=best_global;best=-1
        for th in THRESHOLD_GRID:
            sc=f1_score(y_true[:,j],(y_prob[:,j]>=th).astype('int8'),zero_division=0)
            if sc>best+1e-12 or (abs(sc-best)<=1e-12 and abs(th-best_global)<abs(best_t-best_global)):
                best=sc;best_t=float(th)
        ts[j]=best_t
    global_report=metrics_for(y_true,y_prob,np.full(NUM_LABELS,best_global))
    calibrated_report=metrics_for(y_true,y_prob,ts)
    # Per-label tuning can overfit rare IDs; do not adopt it unless it improves dev Macro-F1.
    if calibrated_report['macro_f1']+1e-12<global_report['macro_f1']:
        ts=np.full(NUM_LABELS,best_global)
        calibrated_report=global_report
    calibrated_report['global_threshold']=best_global
    calibrated_report['per_label_thresholds_used']=int(np.sum(ts!=best_global))
    return ts,calibrated_report

@torch.no_grad()
def infer(model,loader):
    model.eval()
    probs=[];golds=[]
    for ids,mask,cmask,labels in tqdm(loader,leave=False,desc='Inference'):
        ids=ids.to(DEVICE);mask=mask.to(DEVICE);cmask=cmask.to(DEVICE)
        with torch.autocast('cuda',dtype=AMP_DTYPE,enabled=USE_AMP):
            logits=model(ids,mask,cmask)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        golds.append(labels.numpy())
    return np.concatenate(probs),np.concatenate(golds)


### 9. Optimizer, reproducible training, best-checkpoint selection


In [19]:
def make_optimizer(model):
    backbone=[];other=[]
    for n,p in model.named_parameters():
        if not p.requires_grad:continue
        if n.startswith('backbone.') and not ('.A.' in n or '.B.' in n):backbone.append(p)
        else:other.append(p)
    groups=[]
    if backbone:groups.append({'params':backbone,'lr':LR_BACKBONE})
    if other:groups.append({'params':other,'lr':LR_ADAPTER if METHOD in ('lora','lorma') else LR_HEAD})
    return torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)

def save_checkpoint(model,filename,epoch,thresholds,report,model_method=None):
    path=OUTPUT_DIR/filename
    torch.save({'model_state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'model_id':MODEL_ID,'method':model_method or METHOD,'num_labels':NUM_LABELS,
                'label_ids':label_ids,'epoch':epoch,'metrics':report,
                'thresholds':list(map(float,thresholds)),
                'config':{'n_embd':N_EMBD,'n_layer':N_LAYER,'max_chunks':MAX_CHUNKS,
                          'tokens_per_chunk':TOKENS_PER_CHUNK,'student_layers':STUDENT_LAYERS}},path)
    print('Saved:',path)
    return path

def train_one(model,epochs=EPOCHS,teacher=None,model_method=None,ckpt_name='best_model.pt'):
    model.to(DEVICE)
    if teacher is not None:teacher.to(DEVICE).eval().requires_grad_(False)
    opt=make_optimizer(model)
    total_steps=max(1,math.ceil(len(train_dl)/GRAD_ACCUM)*epochs)
    warmup=max(1,int(WARMUP_FRACTION*total_steps))
    scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lambda step:
       min((step+1)/warmup,max(0.,(total_steps-step)/max(1,total_steps-warmup))))
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
    best=-1.;best_path=None;bad_epochs=0;history=[]
    for epoch in range(epochs):
        model.train(); running=0.;n=0;opt.zero_grad(set_to_none=True)
        bar=tqdm(train_dl,desc=f'Epoch {epoch+1}/{epochs}',leave=False)
        for step,(ids,mask,cmask,y) in enumerate(bar):
            ids=ids.to(DEVICE);mask=mask.to(DEVICE);cmask=cmask.to(DEVICE);y=y.to(DEVICE)
            with torch.autocast('cuda',dtype=AMP_DTYPE,enabled=USE_AMP):
                logits=model(ids,mask,cmask)
                loss=multilabel_loss(logits,y)
                if teacher is not None:
                    with torch.no_grad():teacher_logits=teacher(ids,mask,cmask)
                    # Sigmoid multi-label KD, not softmax categorical KD.
                    t=KD_TEMPERATURE
                    soft=F.binary_cross_entropy_with_logits((logits/t).float(),
                                                            torch.sigmoid((teacher_logits/t).float()))*(t*t)
                    loss=(1-KD_SOFT_WEIGHT)*loss+KD_SOFT_WEIGHT*soft
            if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite loss at epoch {epoch+1}, batch {step}')
            group_start=(step//GRAD_ACCUM)*GRAD_ACCUM
            group_batches=min(GRAD_ACCUM,len(train_dl)-group_start)
            scaler.scale(loss/group_batches).backward()
            running+=float(loss.item());n+=1
            if (step+1)%GRAD_ACCUM==0 or step+1==len(train_dl):
                scaler.unscale_(opt)
                norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
                if not torch.isfinite(norm):raise FloatingPointError('Nonfinite gradient norm')
                old_scale=scaler.get_scale()
                scaler.step(opt);scaler.update()
                # On FP16 overflow, optimizer step is skipped: schedule must not advance.
                if not scaler.is_enabled() or scaler.get_scale()>=old_scale:scheduler.step()
                opt.zero_grad(set_to_none=True)
            if step%50==0:bar.set_postfix(loss=f'{running/n:.4f}')
        probs,gt=infer(model,dev_dl)
        thresholds,report=tune_thresholds(gt,probs)
        report.update(epoch=epoch+1,train_loss=running/max(n,1))
        history.append(report)
        print('epoch',epoch+1,report)
        if report['macro_f1']>best+1e-8:
            best=report['macro_f1'];bad_epochs=0
            best_path=save_checkpoint(model,ckpt_name,epoch+1,thresholds,report,model_method)
        else:
            bad_epochs+=1
            if bad_epochs>=PATIENCE:break
        (OUTPUT_DIR/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
    if best_path is None:raise RuntimeError('No best model was saved')
    bundle=torch.load(best_path,map_location='cpu',weights_only=True)
    model.load_state_dict(bundle['model_state_dict'],strict=True)
    model.to(DEVICE).eval()
    return model,bundle,history


### 10. Build model and perform forward/backward preflight


In [20]:
# A same-model-ID, same-label-order, already trained Phase-5 LoRMA teacher is mandatory.
def choose_teacher_path(explicit):
    if explicit:
        p=Path(explicit)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    candidates=[]
    for root in (Path('/kaggle/input'),Path('/kaggle/working')):
        if root.exists():
            candidates.extend(root.rglob('best_model.pt'))
    matched=[p for p in candidates if f'phase5_lsi_model{MODEL_ID}_lorma' in str(p).lower()]
    if len(matched)!=1:
        raise FileNotFoundError('KD requires a Phase-5 LoRMA teacher checkpoint for this Model ID. '
                                'First run its LORMA notebook, attach best_model.pt, then set '
                                f'TEACHER_CKPT_PATH explicitly. Matches: {[str(p) for p in matched]}')
    return matched[0]

teacher_path=choose_teacher_path(TEACHER_CKPT_PATH)
teacher_ckpt=torch.load(teacher_path,map_location='cpu',weights_only=True)
assert teacher_ckpt.get('model_id')==MODEL_ID, 'Teacher model architecture mismatch'
assert teacher_ckpt.get('method')=='lorma', 'Teacher must be fine-tuned Phase-5 LoRMA, not Phase-1'
assert teacher_ckpt.get('label_ids')==label_ids, 'Teacher statute-label order mismatch'
assert teacher_ckpt.get('num_labels')==NUM_LABELS, 'Teacher dimension mismatch'
conf=teacher_ckpt.get('config',{})
assert conf.get('max_chunks')==MAX_CHUNKS and conf.get('tokens_per_chunk')==TOKENS_PER_CHUNK, (
    'Teacher preprocessing/chunk shapes differ; align Phase-5 configs before distillation.')
teacher=MultiLabelLegalClassifier(new_base(),NUM_LABELS,method='lorma')
teacher.load_state_dict(teacher_ckpt['model_state_dict'],strict=True)
teacher.to(DEVICE).eval().requires_grad_(False)
# Drop upper layers while preserving first-layer indices. Model 4 odd/even FFN layouts remain aligned.
student=MultiLabelLegalClassifier(new_base(STUDENT_LAYERS),NUM_LABELS,method='student_full').to(DEVICE)
print('Teacher params',sum(p.numel() for p in teacher.parameters()),
      '| Student params',sum(p.numel() for p in student.parameters()))
# Validate a complete student/teacher forward pass and gradients.
ids,mask,cmask,y=next(iter(train_dl))
ids=ids.to(DEVICE);mask=mask.to(DEVICE);cmask=cmask.to(DEVICE);y=y.to(DEVICE)
with torch.no_grad():teacher_output=teacher(ids,mask,cmask)
student_output=student(ids,mask,cmask)
assert teacher_output.shape==student_output.shape==(len(ids),NUM_LABELS)
loss=multilabel_loss(student_output,y)
loss.backward()
assert any(p.grad is not None for p in student.parameters() if p.requires_grad)
student.zero_grad(set_to_none=True)
print('KD preflight OK',teacher_output.shape)
del ids,mask,cmask,y,teacher_output,student_output,loss
gc.collect();torch.cuda.empty_cache() if DEVICE.type=='cuda' else None


Teacher params 53734900 | Student params 36232660
KD preflight OK torch.Size([4, 100])


### 11. Train and choose the best public DEV checkpoint


In [21]:
student,best_bundle,history=train_one(student,teacher=teacher,model_method='student_full')
model=student
thresholds=np.array(best_bundle['thresholds'],dtype=float)
print('Best student DEV metrics:',best_bundle['metrics'])


Epoch 1/3:   0%|          | 0/10688 [00:00<?, ?it/s]

Inference:   0%|          | 0/2546 [00:00<?, ?it/s]

epoch 1 {'macro_f1': 0.18845719069403127, 'micro_f1': 0.25595019405347996, 'subset_accuracy': 9.822217856792063e-05, 'macro_precision': 0.1598356943850159, 'macro_recall': 0.3395629707392513, 'predicted_labels_per_case': 10.788232983007562, 'global_threshold': 0.3500000000000001, 'per_label_thresholds_used': 93, 'epoch': 1, 'train_loss': 1.3472983962605278}
Saved: /kaggle/working/phase5_lsi_model4_kd_reference/best_model.pt


Epoch 2/3:   0%|          | 0/10688 [00:00<?, ?it/s]

Inference:   0%|          | 0/2546 [00:00<?, ?it/s]

epoch 2 {'macro_f1': 0.21213286946996873, 'micro_f1': 0.28484539057470754, 'subset_accuracy': 0.0005893330714075238, 'macro_precision': 0.18842329580511566, 'macro_recall': 0.3342535715947128, 'predicted_labels_per_case': 9.358707396130047, 'global_threshold': 0.45000000000000007, 'per_label_thresholds_used': 93, 'epoch': 2, 'train_loss': 1.3116145118357179}
Saved: /kaggle/working/phase5_lsi_model4_kd_reference/best_model.pt


Epoch 3/3:   0%|          | 0/10688 [00:00<?, ?it/s]

Inference:   0%|          | 0/2546 [00:00<?, ?it/s]

epoch 3 {'macro_f1': 0.21761604316103458, 'micro_f1': 0.28997174712154816, 'subset_accuracy': 0.0009822217856792063, 'macro_precision': 0.18436097895646392, 'macro_recall': 0.34322756204656385, 'predicted_labels_per_case': 9.01699243689225, 'global_threshold': 0.3500000000000001, 'per_label_thresholds_used': 91, 'epoch': 3, 'train_loss': 1.3028205540923479}
Saved: /kaggle/working/phase5_lsi_model4_kd_reference/best_model.pt
Best student DEV metrics: {'macro_f1': 0.21761604316103458, 'micro_f1': 0.28997174712154816, 'subset_accuracy': 0.0009822217856792063, 'macro_precision': 0.18436097895646392, 'macro_recall': 0.34322756204656385, 'predicted_labels_per_case': 9.01699243689225, 'global_threshold': 0.3500000000000001, 'per_label_thresholds_used': 91, 'epoch': 3, 'train_loss': 1.3028205540923479}


### 12. Re-load best checkpoint and generate Codabench submission with EXACT test IDs


In [22]:
# Reload the actual best checkpoint before prediction, including exact tensor shapes.
check=torch.load(OUTPUT_DIR/'best_model.pt',map_location='cpu',weights_only=True)
assert check['model_id']==MODEL_ID and check['label_ids']==label_ids
assert check['num_labels']==NUM_LABELS
model.load_state_dict(check['model_state_dict'],strict=True)
model.to(DEVICE).eval()
thresholds=np.asarray(check['thresholds'],dtype=float)
assert thresholds.shape==(NUM_LABELS,) and np.isfinite(thresholds).all()

dev_probs,dev_gold=infer(model,dev_dl)
report=metrics_for(dev_gold,dev_probs,thresholds)
print('Re-loaded best checkpoint / DEV:',report)
assert abs(report['macro_f1']-check['metrics']['macro_f1'])<1e-5, (
    'Checkpoint validation Macro-F1 changed after reload: investigate before submission')

test_probs,_=infer(model,test_dl)
assert test_probs.shape==(len(test_records),NUM_LABELS)
submission=[]
for row,prob in zip(test_records,test_probs):
    labels=[int(label_ids[j]) for j in np.flatnonzero(prob>=thresholds)]
    # Empty is a legitimate multi-label prediction; never force a false-positive label.
    assert len(labels)==len(set(labels)) and all(type(v)==int and v in label_to_col for v in labels)
    submission.append({'id':row['id'],'pred':labels})
assert len(submission)==len(test_records)
assert [str(x['id']) for x in submission]==[str(x['id']) for x in test_records]
assert len({str(x['id']) for x in submission})==len(submission)
json_path=OUTPUT_DIR/'prediction.json'
json_path.write_text(json.dumps(submission,ensure_ascii=False,indent=2),encoding='utf-8')
with zipfile.ZipFile(OUTPUT_DIR/'submission.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(json_path,arcname='prediction.json')
with zipfile.ZipFile(OUTPUT_DIR/'submission.zip') as archive:
    assert archive.namelist()==['prediction.json']
    back=json.loads(archive.read('prediction.json'))
assert back==submission
(OUTPUT_DIR/'final_dev_metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
(OUTPUT_DIR/'thresholds.json').write_text(json.dumps(dict(zip(map(str,label_ids),map(float,thresholds))),indent=2),encoding='utf-8')
print('Ready:',OUTPUT_DIR/'submission.zip','| test docs',len(submission),
      '| average predicted statute IDs',np.mean([len(x['pred']) for x in submission]))
print('Example prediction:',submission[0] if submission else None)

# Save standalone inference code and the matching tokenizer.
import shutil
definition_constants={k:globals()[k] for k in ['MODEL_ID', 'VOCAB_SIZE', 'N_EMBD', 'N_HEAD', 'N_KV_HEAD', 'MAX_SEQ_LEN', 'ADAPTER_RANK', 'ADAPTER_ALPHA', 'DROPOUT']}
definition_header='EXPECTED_MODEL_ID='+str(MODEL_ID)+'\nCHECKPOINT_GRADIENTS=False\n'+'\n'.join(k+'='+repr(v) for k,v in definition_constants.items())+'\n'
(OUTPUT_DIR/"model_defs.py").write_text(definition_header+"import math\nimport torch\nimport torch.utils.checkpoint\nfrom torch import nn\nimport torch.nn.functional as F\n# Exact Phase-1 Model 4 module/key layout reused from the user's earlier experiment.\nclass RMSNorm(nn.Module):\n    def __init__(self,dim,eps=1e-6):\n        super().__init__(); self.eps=eps; self.weight=nn.Parameter(torch.ones(dim))\n    def forward(self,x):\n        return x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+self.eps)*self.weight\n\ndef precompute_rope(dim,max_len,theta=10000.0):\n    f=1.0/(theta**(torch.arange(0,dim,2).float()/dim))\n    xy=torch.outer(torch.arange(max_len),f)\n    return torch.view_as_real(torch.polar(torch.ones_like(xy),xy))\n\ndef apply_rope(q,k,f):\n    qc=torch.view_as_complex(q.float().reshape(*q.shape[:-1],-1,2))\n    kc=torch.view_as_complex(k.float().reshape(*k.shape[:-1],-1,2))\n    z=f.view(1,qc.shape[1],1,qc.shape[-1])\n    return torch.view_as_real(qc*z).flatten(3).type_as(q),torch.view_as_real(kc*z).flatten(3).type_as(k)\n\nclass SinkWindowAttention(nn.Module):\n    def __init__(self,head_dim,window_size=256,sink_size=4):\n        super().__init__();self.scale=head_dim**(-0.5)\n        self.window_size=window_size;self.sink_size=sink_size\n    def create_sink_window_mask(self,Lq,Lk,device):\n        row=torch.arange(Lq,device=device).unsqueeze(1)\n        col=torch.arange(Lk,device=device).unsqueeze(0)\n        m=(col<=row)&((row-col)<self.window_size)\n        if self.sink_size:m=m|((col<self.sink_size)&(col<=row))\n        return m[None,None,:,:]\n    def forward(self,q,k,v,is_causal=True,attention_mask=None):\n        Lq,Lk=q.shape[2],k.shape[2]\n        scores=(q@k.transpose(-2,-1))*self.scale\n        if is_causal and Lq>1:\n            scores=scores.masked_fill(~self.create_sink_window_mask(Lq,Lk,q.device),\n                                       torch.finfo(scores.dtype).min)\n        if attention_mask is not None:\n            scores=scores.masked_fill(~attention_mask[:,None,None,:].bool(),torch.finfo(scores.dtype).min)\n        return torch.softmax(scores,dim=-1)@v\n\nclass EfficientGQA(nn.Module):\n    def __init__(self,dim,n_heads,n_kv_heads,window_size=256,sink_size=4):\n        super().__init__()\n        self.n_heads=n_heads;self.n_kv_heads=n_kv_heads;self.head_dim=dim//n_heads\n        self.wq=nn.Linear(dim,n_heads*self.head_dim,bias=False)\n        self.wk=nn.Linear(dim,n_kv_heads*self.head_dim,bias=False)\n        self.wv=nn.Linear(dim,n_kv_heads*self.head_dim,bias=False)\n        self.wo=nn.Linear(dim,dim,bias=False)\n        self.attention=SinkWindowAttention(self.head_dim,window_size,sink_size)\n    def forward(self,x,freqs_cis,kv_cache=None,attention_mask=None):\n        B,L,_=x.shape\n        q=self.wq(x).view(B,L,self.n_heads,self.head_dim)\n        k=self.wk(x).view(B,L,self.n_kv_heads,self.head_dim)\n        v=self.wv(x).view(B,L,self.n_kv_heads,self.head_dim)\n        q,k=apply_rope(q,k,freqs_cis)\n        if kv_cache is not None:\n            kc,vc=kv_cache;k=torch.cat((kc,k),dim=1);v=torch.cat((vc,v),dim=1)\n        new_cache=(k,v)\n        k=k.repeat_interleave(self.n_heads//self.n_kv_heads,dim=2)\n        v=v.repeat_interleave(self.n_heads//self.n_kv_heads,dim=2)\n        out=self.attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),\n                           is_causal=(L>1),attention_mask=attention_mask)\n        return self.wo(out.transpose(1,2).contiguous().view(B,L,-1)),new_cache\n\nclass StandardFFN(nn.Module):\n    def __init__(self,dim):\n        super().__init__();self.net=nn.Sequential(nn.Linear(dim,dim*4,bias=False),nn.GELU(),\n                                                  nn.Linear(dim*4,dim,bias=False))\n    def forward(self,x):return self.net(x)\nclass BottleneckFFN(nn.Module):\n    def __init__(self,dim):\n        super().__init__();self.net=nn.Sequential(nn.Linear(dim,dim//4,bias=False),nn.GELU(),\n                                                  nn.Linear(dim//4,dim,bias=False))\n    def forward(self,x):return self.net(x)\nclass EfficientTransformerBlock(nn.Module):\n    def __init__(self,dim,n_heads,n_kv_heads,window_size=256,sink_size=4,is_bottleneck=False):\n        super().__init__()\n        self.norm1=RMSNorm(dim);self.norm2=RMSNorm(dim)\n        self.attn=EfficientGQA(dim,n_heads,n_kv_heads,window_size,sink_size)\n        self.ffn=BottleneckFFN(dim) if is_bottleneck else StandardFFN(dim)\n    def forward(self,x,freqs_cis,kv_cache=None,attention_mask=None):\n        out,c=self.attn(self.norm1(x),freqs_cis,kv_cache,attention_mask)\n        h=x+out\n        return h+self.ffn(self.norm2(h)),c\nclass CustomLegalModel(nn.Module):\n    def __init__(self,vocab_size,dim,n_layers,n_heads,n_kv_heads,max_len,window_size=256,sink_size=4):\n        super().__init__()\n        self.embeddings=nn.Embedding(vocab_size,dim)\n        self.layers=nn.ModuleList([\n            EfficientTransformerBlock(dim,n_heads,n_kv_heads,window_size,sink_size,i%2!=0)\n            for i in range(n_layers)])\n        self.norm=RMSNorm(dim)\n        self.output=nn.Linear(dim,vocab_size,bias=False)\n        self.output.weight=self.embeddings.weight\n        self.register_buffer('freqs_cis',precompute_rope(dim//n_heads,max_len))\n    def forward(self,tokens,start_pos=0,kv_caches=None,attention_mask=None):\n        _,L=tokens.shape\n        freqs=torch.view_as_complex(self.freqs_cis[start_pos:start_pos+L].to(tokens.device))\n        x=self.embeddings(tokens);caches=[]\n        for i,layer in enumerate(self.layers):\n            x,c=layer(x,freqs,kv_caches[i] if kv_caches is not None else None,attention_mask)\n            caches.append(c)\n        return self.output(self.norm(x)),caches\n\nclass LowRankAdditive(nn.Module):\n    # LoRA: delta W = B @ A, evaluated on the input of frozen W0.\n    def __init__(self,base,r=8,alpha=16,dropout=0.05):\n        super().__init__();self.base=base;self.base.requires_grad_(False)\n        self.A=nn.Linear(base.in_features,r,bias=False)\n        self.B=nn.Linear(r,base.out_features,bias=False)\n        self.drop=nn.Dropout(dropout);self.scale=alpha/r\n        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5));nn.init.zeros_(self.B.weight)\n    def forward(self,x):return self.base(x)+self.scale*self.B(self.A(self.drop(x)))\n\nclass LowRankMultiplicative(nn.Module):\n    # LORMA-style post-projection low-rank multiplicative update:\n    # (I + alpha/r * BA) W0 x, distinct from additive LoRA on x.\n    def __init__(self,base,r=8,alpha=16,dropout=0.05):\n        super().__init__();self.base=base;self.base.requires_grad_(False)\n        self.A=nn.Linear(base.out_features,r,bias=False)\n        self.B=nn.Linear(r,base.out_features,bias=False)\n        self.drop=nn.Dropout(dropout);self.scale=alpha/r\n        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5));nn.init.zeros_(self.B.weight)\n    def forward(self,x):\n        y=self.base(x);return y+self.scale*self.B(self.A(self.drop(y)))\n\ndef adapt_backbone(backbone,method):\n    if method in ('full','student_full'):\n        backbone.requires_grad_(True)\n        return\n    if method not in ('lora','lorma'):\n        raise ValueError('Expected full, lora, lorma or student_full; got '+method)\n    backbone.requires_grad_(False)\n    target=LowRankMultiplicative if method=='lorma' else LowRankAdditive\n    for block in backbone.layers:\n        for key in ('wq','wk','wv','wo'):\n            original=getattr(block.attn,key)\n            setattr(block.attn,key,target(original,ADAPTER_RANK,ADAPTER_ALPHA,DROPOUT))\n\nclass MultiLabelLegalClassifier(nn.Module):\n    def __init__(self,backbone,num_labels,method='full'):\n        super().__init__()\n        self.backbone=backbone\n        adapt_backbone(self.backbone,method)\n        # Pool masked tokens -> mean of chunk representations -> multi-label logits.\n        self.head=nn.Sequential(nn.LayerNorm(N_EMBD*2),nn.Dropout(DROPOUT),\n                                nn.Linear(N_EMBD*2,num_labels))\n        self.num_labels=num_labels\n    def forward(self,input_ids,token_mask,chunk_mask):\n        B,K,L=input_ids.shape\n        ids=input_ids.view(B*K,L)\n        mask=token_mask.view(B*K,L)\n        base=self.backbone\n        freqs=torch.view_as_complex(base.freqs_cis[:L].to(ids.device))\n        x=(base.tok_embeddings(ids) if MODEL_ID==2 else base.embeddings(ids))\n        for layer in base.layers:\n            if CHECKPOINT_GRADIENTS and self.training and x.requires_grad:\n                def call(h,layer=layer):\n                    return layer(h,freqs,attention_mask=mask)[0]\n                x=torch.utils.checkpoint.checkpoint(call,x,use_reentrant=False)\n            else:\n                x,_=layer(x,freqs,attention_mask=mask)\n        x=base.norm(x)\n        active=mask.unsqueeze(-1).to(x.dtype)\n        mean=(x*active).sum(1)/active.sum(1).clamp(min=1)\n        # For right-padded sequences: final valid token (causally aware document vector).\n        final=(mask.long().sum(1)-1).clamp(min=0)\n        last=x[torch.arange(x.size(0),device=x.device),final]\n        per_chunk=torch.cat((mean,last),dim=-1).view(B,K,-1)\n        weights=chunk_mask.to(per_chunk.dtype).unsqueeze(-1)\n        doc=(per_chunk*weights).sum(1)/weights.sum(1).clamp(min=1)\n        return self.head(doc)\n\ndef load_model(path,device='cpu'):\n    obj=torch.load(path,map_location='cpu',weights_only=True)\n    if obj.get('model_id') != EXPECTED_MODEL_ID:\n        raise ValueError('Wrong backbone checkpoint')\n    cfg=obj['config']\n    global NUM_LABELS, N_EMBD\n    NUM_LABELS=obj['num_labels'];N_EMBD=cfg['n_embd']\n    layers=cfg['student_layers'] if obj['method']=='student_full' else cfg['n_layer']\n    base=CustomLegalModel(VOCAB_SIZE,N_EMBD,layers,N_HEAD,N_KV_HEAD,MAX_SEQ_LEN)\n    model=MultiLabelLegalClassifier(base,NUM_LABELS,method=obj['method'])\n    model.load_state_dict(obj['model_state_dict'],strict=True)\n    return model.to(device).eval(),obj\n",encoding="utf-8")
shutil.copy2(tokenizer_path,OUTPUT_DIR/"custom_bpe_scratch.json")


Inference:   0%|          | 0/2546 [00:00<?, ?it/s]

Re-loaded best checkpoint / DEV: {'macro_f1': 0.21761604316103458, 'micro_f1': 0.28997174712154816, 'subset_accuracy': 0.0009822217856792063, 'macro_precision': 0.18436097895646392, 'macro_recall': 0.34322756204656385, 'predicted_labels_per_case': 9.01699243689225}


Inference:   0%|          | 0/3255 [00:00<?, ?it/s]

Ready: /kaggle/working/phase5_lsi_model4_kd_reference/submission.zip | test docs 13019 | average predicted statute IDs 8.592749059067517
Example prediction: {'id': '100002997', 'pred': [5, 6, 26, 34, 35, 37, 38, 41, 94]}


PosixPath('/kaggle/working/phase5_lsi_model4_kd_reference/custom_bpe_scratch.json')